In [251]:
import pandas as pd
import altair as alt
from altair import layer
from pandas import to_datetime

# Charts that show the breakdown of a total

In [252]:
austria_election = pd.DataFrame({
    "category": ["ÖVP", "SPÖ", "FPÖ", "Grüne", "NEOS", "Bierpartei", "Others"],
    "value": [26.3,  21.1, 28.8,  8.2, 9.1, 2.0, 4.5]
})
austria_election

,category,value
0,ÖVP,26.3
1,SPÖ,21.1
2,FPÖ,28.8
3,Grüne,8.2
4,NEOS,9.1
5,Bierpartei,2.0
6,Others,4.5


## Donut Chart

In [253]:
alt.Chart(austria_election).mark_arc(innerRadius=50).encode(
    theta= alt.Theta("value:Q"),
    color=alt.Color("category:N")
)

alt.Chart(...)

## Pie Chart

In [254]:
base = alt.Chart(austria_election).encode(
    alt.Theta("value:Q").stack(True),
    alt.Color("category:N").legend(None)
)

pie = base.mark_arc(outerRadius=120)
text = base.mark_text(radius=140, size=15).encode(text="category:N")

pie + text

alt.LayerChart(...)

# Chart for Temporal Data

In [255]:
from vega_datasets import data
stocks = data.stocks()
stocks

,symbol,date,price
0,MSFT,2000-01-01,39.81
1,MSFT,2000-02-01,36.35
2,MSFT,2000-03-01,43.22
3,MSFT,2000-04-01,28.37
4,MSFT,2000-05-01,25.45
...,...,...,...
555,AAPL,2009-11-01,199.91
556,AAPL,2009-12-01,210.73
557,AAPL,2010-01-01,192.06
558,AAPL,2010-02-01,204.62


In [256]:
alt.Chart(stocks).mark_line().encode(
    x='date:T',
    y='price:Q',
    color='symbol'
)

alt.Chart(...)

## Area Chart

In [257]:
alt.Chart(stocks).transform_filter(
    'datum.symbol==="GOOG"'
).mark_area().encode(
    alt.X('date:T'),
    alt.Y('price:Q')
)

alt.Chart(...)

## Stacked Area Chart

In [258]:
alt.Chart(stocks).mark_area().encode(
    x="date:T",
    y="price:Q",
    color="symbol:N"
)


alt.Chart(...)

## Slope Chart

In [259]:
first_last_stocks = stocks.groupby('symbol').apply(lambda x: pd.concat([x.head(1), x.tail(1)])).reset_index(level=0)
first_last_stocks = first_last_stocks[first_last_stocks['symbol'] != 'GOOG']
first_last_stocks

,symbol,date,price
437,AAPL,2000-01-01,25.94
559,AAPL,2010-03-01,223.02
123,AMZN,2000-01-01,64.56
245,AMZN,2010-03-01,128.82
246,IBM,2000-01-01,100.52
368,IBM,2010-03-01,125.55
0,MSFT,2000-01-01,39.81
122,MSFT,2010-03-01,28.80


In [260]:
alt.Chart(first_last_stocks).mark_line().encode(
    x='date:O',
    y='price',
    color='symbol'
)

alt.Chart(...)

In [261]:
first_last_stocks['year'] = pd.DatetimeIndex(first_last_stocks['date']).year
first_last_stocks

,symbol,date,price,year
437,AAPL,2000-01-01,25.94,2000
559,AAPL,2010-03-01,223.02,2010
123,AMZN,2000-01-01,64.56,2000
245,AMZN,2010-03-01,128.82,2010
246,IBM,2000-01-01,100.52,2000
368,IBM,2010-03-01,125.55,2010
0,MSFT,2000-01-01,39.81,2000
122,MSFT,2010-03-01,28.80,2010


In [262]:
alt.Chart(first_last_stocks).mark_line().encode(
    x='year:O',
    y='price',
    color='symbol'
).properties(
    width=300
)

alt.Chart(...)

# Charts for Spatial Data

In [263]:
airports = data.airports()
airports.head()

,iata,name,city,state,country,latitude,longitude
0,00M,Thigpen,Bay Springs,MS,USA,31.953765,-89.234505
1,00R,Livingston Municipal,Livingston,TX,USA,30.685861,-95.017928
2,00V,Meadow Lake,Colorado Springs,CO,USA,38.945749,-104.569893
3,01G,Perry-Warsaw,Perry,NY,USA,42.741347,-78.052081
4,01J,Hilliard Airpark,Hilliard,FL,USA,30.688012,-81.905944


The chart is quite similar to a standard scatterplot, with a few differences:

We specify the encodings “latitude” and “longitude” instead of “x” and “y” and we specify a projection to be used for the data. For data covering only the United States, the projection “albersUsa” is useful:

In [264]:
alt.Chart(airports).mark_circle().encode(
    longitude='longitude:Q',
    latitude='latitude:Q',
    size=alt.value(7),
    tooltip='name'
).project(
    "albersUsa"
).properties(
    width=400,
    height=300
)

alt.Chart(...)

All available projections are shown in the [Vega Documentation](https://vega.github.io/vega/docs/projections/) and in the [Vega-Lite Docuemtantion](https://vega.github.io/vega-lite/docs/projection.html).

## Background maps

If a geographical map with states and country borders is to be displayed, then a geographical shape file must be loaded into Altair and the ``geoshape`` marker must be used:

In [265]:
states = alt.topo_feature(data.us_10m.url, feature='states')

alt.Chart(states).mark_geoshape(
    fill='lightgray',
    stroke='white'
).project('albersUsa').properties(
    width=500,
    height=300
)

alt.Chart(...)

And here are the world's national borders:

In [266]:
countries = alt.topo_feature(data.world_110m.url, 'countries')

alt.Chart(countries).mark_geoshape(
    fill='lightgray',
    stroke='white'
).project(
    "equirectangular"
).properties(
    width=500,
    height=300
)

alt.Chart(...)

You can see what happens if you try other projection types, e.g. ``equirectangular``, ``mercator``, ``orthographic``, ``albers``, oder ``gnomonic``

### Different projections:

In [267]:
source = alt.topo_feature(data.world_110m.url, 'countries')

base = alt.Chart(source).mark_geoshape(
    fill='#666666',
    stroke='white'
).properties(
    width=300,
    height=180
)

projections = ['equirectangular', 'mercator', 'orthographic', 'gnomonic'] # go through different projects (see Altair docu)
charts = [base.project(proj).properties(title=proj)
          for proj in projections]

alt.concat(*charts, columns=2)

alt.ConcatChart(...)

# Using maps from other countries

In [268]:
#!pip install geopandas gpdvega

In [269]:
import geopandas as gpd
from shapely.geometry import Point, Polygon # in case points and polygons are needed

### Shapefiles

* .shp - shapefile that is actually geometry like states
* .dbf - database with addional infos outside the shape
* .prj - projection file with shape of earth etc.
* .shx - index file (makes it faster)

Austria: https://www.esri-austria.at/service/open-data-und-projektionen-fuer-oesterreich/

https://data-synergis.opendata.arcgis.com/datasets/SynerGIS::bundesl%C3%A4ndergrenzen-150-000-stand-2017/explore?location=47.665927%2C13.346488%2C6.99


In [270]:
# when reading the shape file the other files in the folder will be read automatically
austria = gpd.read_file("shapefiles_austria/Bundeslaender_50.shp")
austria

,OBJECTID,BL_KZ,BL,ST_KZ,ST,FL,FL_KM,STAND,SHAPE_Leng,SHAPE_Area,geometry
0,1,1,Burgenland,1,Österreich,3.965203e+09,3965.202735,2017.10.01,8.537887,0.473661,"POLYGON ((17.07216 48.1149, 17.07278 48.11453,..."
1,2,2,Kärnten,1,Österreich,9.536545e+09,9536.545489,2017.10.01,7.883018,1.123930,"POLYGON ((12.72541 47.13128, 12.72972 47.13091..."
2,3,3,Niederösterreich,1,Österreich,1.917926e+10,19179.264850,2017.10.01,13.346968,2.325575,"POLYGON ((15.02218 49.02115, 15.02242 49.02099..."
3,4,4,Oberösterreich,1,Österreich,1.198259e+10,11982.589835,2017.10.01,10.088962,1.448179,"POLYGON ((13.84668 48.77058, 13.84847 48.76977..."
4,5,5,Salzburg,1,Österreich,7.154584e+09,7154.584478,2017.10.01,8.805660,0.853029,"POLYGON ((13.04706 48.03906, 13.04652 48.03795..."
5,6,6,Steiermark,1,Österreich,1.639876e+10,16398.760502,2017.10.01,10.606761,1.950918,"POLYGON ((15.34408 47.82586, 15.34481 47.82562..."
6,7,7,Tirol,1,Österreich,1.264834e+10,12648.339352,2017.10.01,11.922909,1.501331,"MULTIPOLYGON (((12.48352 47.15784, 12.49147 47..."
7,8,8,Vorarlberg,1,Österreich,2.601657e+09,2601.657495,2017.10.01,3.778444,0.309244,"POLYGON ((9.78041 47.59402, 9.7809 47.59402, 9..."
8,9,9,Wien,1,Österreich,4.147524e+08,414.752412,2017.10.01,1.558044,0.050201,"POLYGON ((16.43804 48.31626, 16.4383 48.31448,..."


In [271]:
base_austria = alt.Chart(austria).mark_geoshape(
    fill='lightgrey',
    stroke='#AAAAAA'
).properties(
    width=300,
    height=180
)
base_austria

alt.Chart(...)

**Filtern einzelner Bundesländer:**

In [272]:
alt.Chart(austria[austria.SHAPE_Area > 1.5]).mark_geoshape(
    fill='darkblue',
    stroke='#AAAAAA'
).properties(
    width=300,
    height=180
)

alt.Chart(...)

Auch hier können verschiedene Projektionen angewandt werden:

In [273]:
alt.Chart(austria).mark_geoshape(
    fill="#EEEEEE", stroke="darkgray"
).project('conicEqualArea').properties(
    width=500,
    height=300
)

alt.Chart(...)

In [274]:
alt.Chart(austria).mark_geoshape(
    fill="#EEEEEE", stroke="darkgray"
).project('conicConformal')

alt.Chart(...)

## Dots on a Map

In Altair, two charts can be placed on top of each other using '+'

In [275]:
states = alt.topo_feature(data.us_10m.url, feature='states')
airports = data.airports()

background = alt.Chart(states).mark_geoshape(
    fill='lightgray',
    stroke='white'
).project('albersUsa').properties(
    width=500,
    height=300
)

points = alt.Chart(airports).mark_circle().encode(
    longitude='longitude:Q',
    latitude='latitude:Q',
    size=alt.value(8),
    tooltip='name'
)

background + points

alt.LayerChart(...)

### Load data set from Austria and display on map:

**Example Data Set:** Police stations in Austria

https://www.data.gv.at/katalog/dataset/4e5c9414-2886-462e-b19f-a598ad065522#resources

In [276]:
police = pd.read_csv('policestations_austria.csv', sep=";")
police

,OID_,ID,ADRCD,SUBCD,LandID,Land,BezirkID,Bezirk,ArtLang,NameLang,...,Hnr,Telefon,IntVorwahl,NatVorwahl,TelefonNummer,Durchwahl,FaxDurchwahl,EMailAdresse,X,Y
0,1,1428,5817611.0,1.0,6,Steiermark,622,Hartberg-Fürstenfeld,Polizeiinspektion,Ilz,...,38,+43 59133 6222 100,43,59133,6222,100.0,NaN,PI-ST-Ilz@polizei.gv.at,"15,923626909999999","47,086669919999999"
1,2,506,5212254.0,1.0,1,Burgenland,107,Neusiedl am See,Polizeiinspektion,Kittsee,...,6,+43 59133 1142 100,43,59133,1142,100.0,109.0,PI-B-Kittsee@polizei.gv.at,"17,070291529999999","48,092277529999997"
2,3,446,7063018.0,2.0,1,Burgenland,101,Eisenstadt (Stadt),Landespolizeidirektion,Burgenland,...,84,+43 59133 10,43,59133,10,NaN,NaN,LPD-B@polizei.gv.at,"16,542800000000000","47,845700000000001"
3,4,454,5153854.0,1.0,1,Burgenland,101,Eisenstadt (Stadt),Stadtpolizeikommando/Bezirkspolizeikommando,Eisenstadt/Eisenstadt-Umgebung,...,124,+43 59133 15 1823,43,59133,15,1823.0,NaN,SPK-B-Eisenstadt@polizei.gv.at,"16,530600000000000","47,829599999999999"
4,5,455,7063018.0,2.0,1,Burgenland,101,Eisenstadt (Stadt),Polizeiinspektion,Eisenstadt-Neusiedlerstraße,...,84/4,+43 59133 1581 100,43,59133,1581,100.0,109.0,PI-B-Eisenstadt@polizei.gv.at,"16,542800000000000","47,845700000000001"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
928,929,589,6042442.0,1.0,2,Kärnten,201,Klagenfurt (Stadt),Polizeiinspektion,Klagenfurt-St.-Ruprechter-Straße,...,3,+43 59133 2585 100,43,59133,2585,100.0,109.0,PI-K-Klagenfurt-St-Ruprechterstrasse@polizei.g...,"14,307277680000000","46,619857789999998"
929,930,590,7104715.0,1.0,2,Kärnten,201,Klagenfurt (Stadt),Polizeiinspektion,Klagenfurt-Viktring,...,124,+43 59133 2586 100,43,59133,2586,100.0,109.0,PI-K-Klagenfurt-Viktring@polizei.gv.at,"14,271588330000000","46,592372900000001"
930,931,591,6044325.0,1.0,2,Kärnten,201,Klagenfurt (Stadt),Polizeiinspektion,Klagenfurt-Villacher Straße-Seedienst,...,137,+43 59133 2587 100,43,59133,2587,100.0,109.0,PI-K-Klagenfurt-Villacher-Strasse@polizei.gv.at,"14,283369069999999","46,620052340000001"
931,932,593,6038740.0,1.0,2,Kärnten,201,Klagenfurt (Stadt),Verkehrsinspektion,Klagenfurt,...,33H,+43 59133 2595,43,59133,2595,NaN,NaN,VI-K-Klagenfurt@polizei.gv.at,"14,314094550000000","46,632110599999997"


In [277]:
police.info()

<class 'pandas.DataFrame'>
RangeIndex: 933 entries, 0 to 932
Data columns (total 23 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   OID_           933 non-null    int64  
 1   ID             933 non-null    int64  
 2   ADRCD          931 non-null    float64
 3   SUBCD          928 non-null    float64
 4   LandID         933 non-null    int64  
 5   Land           933 non-null    str    
 6   BezirkID       933 non-null    int64  
 7   Bezirk         933 non-null    str    
 8   ArtLang        933 non-null    str    
 9   NameLang       933 non-null    str    
 10  Plz            933 non-null    int64  
 11  Ort            933 non-null    str    
 12  Strasse        933 non-null    str    
 13  Hnr            924 non-null    str    
 14  Telefon        933 non-null    str    
 15  IntVorwahl     933 non-null    int64  
 16  NatVorwahl     933 non-null    int64  
 17  TelefonNummer  933 non-null    int64  
 18  Durchwahl      868 no

Longitude and latitude are still strings and still need to be converted to floats:

In [278]:
police["X"] = police["X"].str.replace(',', '.').astype(float)
police["Y"] = police["Y"].str.replace(',', '.').astype(float)

In [279]:
alt.Chart(police).mark_circle(opacity=0.7).encode(
    # point latitude & longitude to coordinates in X & Y columns
    longitude='X:Q',
    latitude='Y:Q',
    tooltip=['ArtLang', 'Bezirk'],
    color=alt.Color('Land:N', scale=alt.Scale(scheme='dark2')) #color scales: https://vega.github.io/vega/docs/schemes/
)

alt.Chart(...)

Arrange points on map:

In [280]:
# 1.  mark_geoshape transparently uses the geometry column
basemap = alt.Chart(austria).mark_geoshape(
    # add some styling to reduce the salience of the basemap
    fill="#444444", stroke="lightgray"
).project('conicConformal').properties(
    width=500,
    height=300
)

# 2.  we use mark_circle to have more control over visual variables
markers = alt.Chart(police).mark_circle(opacity=0.6).encode(
    longitude='X:Q',
    latitude='Y:Q',
    size=alt.value(12),
    tooltip=['ArtLang', 'Bezirk'],
    color=alt.Color('Land:N', scale=alt.Scale(scheme='pastel1')) #color scales: https://vega.github.io/vega/docs/schemes/
)

# combine the two layers
basemap + markers

alt.LayerChart(...)

## Aggregated police stations

In [281]:
df_police = police[['X', 'Y', 'Land']]
df_police

,X,Y,Land
0,15.923627,47.086670,Steiermark
1,17.070292,48.092278,Burgenland
2,16.542800,47.845700,Burgenland
3,16.530600,47.829600,Burgenland
4,16.542800,47.845700,Burgenland
...,...,...,...
928,14.307278,46.619858,Kärnten
929,14.271588,46.592373,Kärnten
930,14.283369,46.620052,Kärnten
931,14.314095,46.632111,Kärnten


In [282]:
df_agg = df_police.groupby('Land').agg({'Land':'count', 'X':'mean', 'Y':'mean'})
df_agg = df_agg.rename(columns={'Land':'stations'}).reset_index()
df_agg

,Land,stations,X,Y
0,Burgenland,54,16.499220,47.583516
1,Kärnten,88,14.052670,46.707653
2,Niederösterreich,219,15.885454,48.226499
3,Oberösterreich,146,13.989466,48.195578
4,Salzburg,59,13.118841,47.571103
5,Steiermark,152,15.265892,47.160744
6,Tirol,78,11.465856,47.282712
7,Vorarlberg,37,9.784693,47.327778
8,Wien,100,16.365502,48.207568


## Symbol Map

In [283]:
basemap = alt.Chart(austria).mark_geoshape(
    fill="#AAAAAA", stroke="lightgray"
).project('conicConformal').properties(
    width=500,
    height=300
)

markers = alt.Chart(df_agg).mark_circle(opacity=0.6).encode(
    longitude='X:Q',
    latitude='Y:Q',
    size=alt.Size('stations:Q'),
    color=alt.value('steelblue'),
    tooltip=['Land:N','stations:Q']
)

basemap + markers

alt.LayerChart(...)

## Choropleth Map

In [284]:
austria_police = austria.merge(df_agg, left_on='BL', right_on='Land')
austria_police

,OBJECTID,BL_KZ,BL,ST_KZ,ST,FL,FL_KM,STAND,SHAPE_Leng,SHAPE_Area,geometry,Land,stations,X,Y
0,1,1,Burgenland,1,Österreich,3.965203e+09,3965.202735,2017.10.01,8.537887,0.473661,"POLYGON ((17.07216 48.1149, 17.07278 48.11453,...",Burgenland,54,16.499220,47.583516
1,2,2,Kärnten,1,Österreich,9.536545e+09,9536.545489,2017.10.01,7.883018,1.123930,"POLYGON ((12.72541 47.13128, 12.72972 47.13091...",Kärnten,88,14.052670,46.707653
2,3,3,Niederösterreich,1,Österreich,1.917926e+10,19179.264850,2017.10.01,13.346968,2.325575,"POLYGON ((15.02218 49.02115, 15.02242 49.02099...",Niederösterreich,219,15.885454,48.226499
3,4,4,Oberösterreich,1,Österreich,1.198259e+10,11982.589835,2017.10.01,10.088962,1.448179,"POLYGON ((13.84668 48.77058, 13.84847 48.76977...",Oberösterreich,146,13.989466,48.195578
4,5,5,Salzburg,1,Österreich,7.154584e+09,7154.584478,2017.10.01,8.805660,0.853029,"POLYGON ((13.04706 48.03906, 13.04652 48.03795...",Salzburg,59,13.118841,47.571103
5,6,6,Steiermark,1,Österreich,1.639876e+10,16398.760502,2017.10.01,10.606761,1.950918,"POLYGON ((15.34408 47.82586, 15.34481 47.82562...",Steiermark,152,15.265892,47.160744
6,7,7,Tirol,1,Österreich,1.264834e+10,12648.339352,2017.10.01,11.922909,1.501331,"MULTIPOLYGON (((12.48352 47.15784, 12.49147 47...",Tirol,78,11.465856,47.282712
7,8,8,Vorarlberg,1,Österreich,2.601657e+09,2601.657495,2017.10.01,3.778444,0.309244,"POLYGON ((9.78041 47.59402, 9.7809 47.59402, 9...",Vorarlberg,37,9.784693,47.327778
8,9,9,Wien,1,Österreich,4.147524e+08,414.752412,2017.10.01,1.558044,0.050201,"POLYGON ((16.43804 48.31626, 16.4383 48.31448,...",Wien,100,16.365502,48.207568


In [285]:
alt.Chart(austria_police).mark_geoshape().encode(
     color='stations:Q',
     tooltip=['Land:N','stations:Q']
).project('conicConformal').properties(
    width=500,
    height=300
)

alt.Chart(...)

# Exercises

###### 🚀 Exercise 1 - Dot Density Map:
Show the petrol stations on the Austrian map. Take only the petrol stations from a single day (e.g. the last day). Use the tooltip to display the name of the petrol station and encode the super or diesel prices using an ordered color scale.

In [286]:
petrol_stations = pd.read_csv("petrol_stations.csv", sep=",")
petrol_stations

,name,Super95,Diesel,zip Code,federal_state,city,date,coordinates_X,coordinates_Y
0,Lagerhaus Tankstelle,1.799,1.819,9640,Kärnten,Kötschach,2022-08-13 17:57:04,13.003241,46.672118
1,Turmöl Quick,1.769,1.780,2091,Niederösterreich,Langau,2022-08-13 17:57:16,15.717036,48.829642
2,Genol Lagerhaus Tankstelle,0.000,1.799,3851,Niederösterreich,Kautzen,2022-08-13 17:57:24,15.240267,48.929879
3,Turmöl Quick,1.714,1.734,4048,Oberösterreich,Puchenau,2022-08-13 17:57:37,14.241236,48.312442
4,AVIA Xpress (Automaten-Tankstelle),1.704,1.712,4563,Oberösterreich,Micheldorf,2022-08-13 17:57:32,14.129979,47.894056
...,...,...,...,...,...,...,...,...,...
25664,Turmöl,0.000,1.898,4910,Oberösterreich,Ried im Innkreis,2022-09-29 07:06:33,13.460580,48.205890
25665,TurboSprit,1.628,1.858,9130,Kärnten,Poggersdorf,2022-09-29 07:05:58,14.449627,46.641242
25666,Turmöl,1.614,1.874,1230,Wien,Wien,2022-09-29 07:07:11,16.319350,48.154766
25667,Turmöl Quick,1.594,1.864,3150,Niederösterreich,Wilhelmsburg,2022-09-29 07:06:18,15.600520,48.100190


In [287]:
petrol_stations.date = pd.to_datetime(petrol_stations.date)

In [288]:
petrol_stations.info()

<class 'pandas.DataFrame'>
RangeIndex: 25669 entries, 0 to 25668
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   name           25628 non-null  str           
 1   Super95        25669 non-null  float64       
 2   Diesel         25669 non-null  float64       
 3   zip Code       25669 non-null  int64         
 4   federal_state  25669 non-null  str           
 5   city           25669 non-null  str           
 6   date           25669 non-null  datetime64[us]
 7   coordinates_X  25669 non-null  float64       
 8   coordinates_Y  25669 non-null  float64       
dtypes: datetime64[us](1), float64(4), int64(1), str(3)
memory usage: 1.8 MB


In [289]:
petrol_stations_last_day = petrol_stations[petrol_stations.date.dt.date == petrol_stations.date.max().date()]
petrol_stations_last_day

,name,Super95,Diesel,zip Code,federal_state,city,date,coordinates_X,coordinates_Y
17552,Lagerhaus Tankstelle,1.749,1.949,9640,Kärnten,Kötschach,2022-12-09 01:58:15,13.003241,46.672118
17553,Genol,1.719,0.000,3580,Niederösterreich,Horn,2022-12-09 01:58:26,15.637407,48.665968
17554,LMEnergy Express,1.728,0.000,5723,Salzburg,Uttendorf,2022-12-09 01:58:52,12.575030,47.281193
17555,Genol Lagerhaus Tankstelle,0.000,1.899,3851,Niederösterreich,Kautzen,2022-12-09 01:58:35,15.240267,48.929879
17556,Turmöl Quick,1.621,1.884,4048,Oberösterreich,Puchenau,2022-12-09 01:58:47,14.241236,48.312442
...,...,...,...,...,...,...,...,...,...
18265,AVIA XPress (unser Service: Luft und Wasser),1.705,1.899,3542,Niederösterreich,Jaidhof,2022-12-09 01:58:28,15.461022,48.525221
18266,TurboSprit,1.688,1.898,9130,Kärnten,Poggersdorf,2022-12-09 01:58:15,14.449627,46.641242
18267,eni,2.199,2.199,9212,Kärnten,Techelsberg am Wörthersee,2022-12-09 01:58:15,14.094225,46.630239
18268,Turmöl Quick,1.647,0.000,3150,Niederösterreich,Wilhelmsburg,2022-12-09 01:58:32,15.600520,48.100190


In [290]:
basemap = alt.Chart(austria).mark_geoshape(
    fill="#AAAAAA", stroke="lightgray"
).project('conicConformal').properties(
    width=500,
    height=300
)

markers = alt.Chart(petrol_stations_last_day).mark_circle(opacity=0.6).encode(
    longitude='coordinates_X:Q',
    latitude='coordinates_Y:Q',
    tooltip=['name'],
    color=alt.Color('Diesel:Q', scale=alt.Scale(scheme='yelloworangered', domain=[1.5, 1.6, 1.7, 1.8, 1.9, 2.0, 2.1, 2.2], type='threshold'), bin=alt.Bin(step=0.05))
)

chart = (basemap + markers).properties(title='Diesel prices')
chart

alt.LayerChart(...)

###### 🚀 Exercise 2 - Choropleth Map or Proportional Symbol Map :
Display the petrol stations on the Austrian map in aggregated form (e.g. in a Choropleth Map or Proporional Symbol Map). Show either the diesel or Super95 price on the map.

How are the prices distributed on average in Austria? Are there differences between east and west? (Attention: all values with 0.0 should be removed beforehand, otherwise the prices will be distorted).

In [291]:
petrol_stations_diesel =  petrol_stations[petrol_stations['Diesel'] > 0]
petrol_stations_diesel

,name,Super95,Diesel,zip Code,federal_state,city,date,coordinates_X,coordinates_Y
0,Lagerhaus Tankstelle,1.799,1.819,9640,Kärnten,Kötschach,2022-08-13 17:57:04,13.003241,46.672118
1,Turmöl Quick,1.769,1.780,2091,Niederösterreich,Langau,2022-08-13 17:57:16,15.717036,48.829642
2,Genol Lagerhaus Tankstelle,0.000,1.799,3851,Niederösterreich,Kautzen,2022-08-13 17:57:24,15.240267,48.929879
3,Turmöl Quick,1.714,1.734,4048,Oberösterreich,Puchenau,2022-08-13 17:57:37,14.241236,48.312442
4,AVIA Xpress (Automaten-Tankstelle),1.704,1.712,4563,Oberösterreich,Micheldorf,2022-08-13 17:57:32,14.129979,47.894056
...,...,...,...,...,...,...,...,...,...
25664,Turmöl,0.000,1.898,4910,Oberösterreich,Ried im Innkreis,2022-09-29 07:06:33,13.460580,48.205890
25665,TurboSprit,1.628,1.858,9130,Kärnten,Poggersdorf,2022-09-29 07:05:58,14.449627,46.641242
25666,Turmöl,1.614,1.874,1230,Wien,Wien,2022-09-29 07:07:11,16.319350,48.154766
25667,Turmöl Quick,1.594,1.864,3150,Niederösterreich,Wilhelmsburg,2022-09-29 07:06:18,15.600520,48.100190


In [292]:
df_petrol = petrol_stations_diesel.groupby('federal_state').agg({'Diesel':'mean', 'coordinates_X':'mean', 'coordinates_Y':'mean'})
df_petrol

,Diesel,coordinates_X,coordinates_Y
federal_state,,,
Burgenland,1.839888,16.482615,47.574936
Kärnten,1.858821,14.068285,46.697964
Niederösterreich,1.846460,15.838292,48.305650
Oberösterreich,1.841720,13.989247,48.235862
Salzburg,1.870776,13.106519,47.512326
Steiermark,1.840542,15.105631,47.121862
Tirol,1.876481,11.550106,47.287324
Vorarlberg,1.864572,9.732555,47.330566
Wien,1.867234,16.366252,48.201650


In [293]:
basemap = alt.Chart(austria).mark_geoshape(
    fill="#AAAAAA", stroke="lightgray"
).project('conicConformal').properties(
    width=500,
    height=300
)

markers = alt.Chart(df_petrol).mark_circle(opacity=0.6).encode(
    longitude='coordinates_X:Q',
    latitude='coordinates_Y:Q',
    size=alt.Size('Diesel:Q'),
    color=alt.value('steelblue'),
    tooltip=['federal_state:N', 'Diesel:Q']
)

basemap + markers


alt.LayerChart(...)

In [294]:
austria_diesel = austria.merge(df_petrol, left_on='BL', right_on='federal_state')
austria_diesel

,OBJECTID,BL_KZ,BL,ST_KZ,ST,FL,FL_KM,STAND,SHAPE_Leng,SHAPE_Area,geometry,Diesel,coordinates_X,coordinates_Y
0,1,1,Burgenland,1,Österreich,3.965203e+09,3965.202735,2017.10.01,8.537887,0.473661,"POLYGON ((17.07216 48.1149, 17.07278 48.11453,...",1.839888,16.482615,47.574936
1,2,2,Kärnten,1,Österreich,9.536545e+09,9536.545489,2017.10.01,7.883018,1.123930,"POLYGON ((12.72541 47.13128, 12.72972 47.13091...",1.858821,14.068285,46.697964
2,3,3,Niederösterreich,1,Österreich,1.917926e+10,19179.264850,2017.10.01,13.346968,2.325575,"POLYGON ((15.02218 49.02115, 15.02242 49.02099...",1.846460,15.838292,48.305650
3,4,4,Oberösterreich,1,Österreich,1.198259e+10,11982.589835,2017.10.01,10.088962,1.448179,"POLYGON ((13.84668 48.77058, 13.84847 48.76977...",1.841720,13.989247,48.235862
4,5,5,Salzburg,1,Österreich,7.154584e+09,7154.584478,2017.10.01,8.805660,0.853029,"POLYGON ((13.04706 48.03906, 13.04652 48.03795...",1.870776,13.106519,47.512326
5,6,6,Steiermark,1,Österreich,1.639876e+10,16398.760502,2017.10.01,10.606761,1.950918,"POLYGON ((15.34408 47.82586, 15.34481 47.82562...",1.840542,15.105631,47.121862
6,7,7,Tirol,1,Österreich,1.264834e+10,12648.339352,2017.10.01,11.922909,1.501331,"MULTIPOLYGON (((12.48352 47.15784, 12.49147 47...",1.876481,11.550106,47.287324
7,8,8,Vorarlberg,1,Österreich,2.601657e+09,2601.657495,2017.10.01,3.778444,0.309244,"POLYGON ((9.78041 47.59402, 9.7809 47.59402, 9...",1.864572,9.732555,47.330566
8,9,9,Wien,1,Österreich,4.147524e+08,414.752412,2017.10.01,1.558044,0.050201,"POLYGON ((16.43804 48.31626, 16.4383 48.31448,...",1.867234,16.366252,48.201650


In [295]:
alt.Chart(austria_diesel).mark_geoshape().encode(
     color='Diesel:Q',
     tooltip=['federal_state:N','Diesel:Q']
).project('conicConformal').properties(
    width=500,
    height=300
)

alt.Chart(...)

Average diesel prices differ slightly from east to west. Western prices are a few cents higher than eastern prices. Vienna being the exception.